### 1.1 Legrange Interpolation

In [47]:
import numpy as np
def LEGRANGE(x, x_points, y_points):
    
    n = len(x_points)
    sum = 0
    for i in range(n):
        prod = 1
        for j in range(n):
            if (i != j) : 
                prod *= (x-x_points[j])/(x_points[i]-x_points[j])
        sum += prod*y_points[i]     
    return sum

x_points = np.array([0,1,2], dtype = np.double)
y_points = np.array([1,2,0], dtype = np.double)
x_interpolate = 1.5
print(LEGRANGE(x_interpolate, x_points, y_points))

1.375


### 1.2 Newton's Divided Differences

In [104]:
def NEWTON(x, x_points, y_points):
    n = len(x_points)
    
    for j in range(1,n):
        for i in range(n-1, j-1, -1):
            y_points[i] = (y_points[i]-y_points[i-1])/(x_points[i]-x_points[i-j])

    prod = y_points[n-1]
    
    for i in range(n-2, -1, -1):
        prod = y_points[i] + (x-x_points[i])*prod
    return prod
x_points = np.array([-1.2, 0.3, 1.1], dtype = np.double)
y_points = np.array([-5.76, -5.61, -3.69], dtype = np.double)
x_interpolate = 0.3
print(NEWTON(x_interpolate, x_points, y_points))

-5.61


### 2.1 Natural Cubic Spline Interpolation

In [12]:
import numpy as np
def get_spline_matrices(x_points, y_points):
    n = len(x_points)
    h = x_points[1:] - x_points[:-1]
    A = np.zeros((n,n))
    b = np.zeros((n,1))

    A[0,0] = 1
    A[n-1,n-1] = 1
    for i in range(1, n-1):
        A[i,i-1] = h[i-1]
        A[i,i] = 2*(h[i-1]+h[i])
        A[i,i+1] = h[i]
        b[i] = 3 * ((y_points[i+1]-y_points[i])/h[i] - (y_points[i]-y_points[i-1])/h[i-1])
        print("A= ", A)
        print("b= ", b.T)
    return np.linalg.inv(A) @ b
x_points = np.array([-1.2, 0.3, 1.1], dtype = np.double)
y_points = np.array([-5.76, -5.61, -3.69], dtype = np.double)
print(get_spline_matrices(x_points, y_points).T)

A=  [[1.  0.  0. ]
 [1.5 4.6 0.8]
 [0.  0.  1. ]]
b=  [[0.  6.9 0. ]]
[[5.10702591e-16 1.50000000e+00 0.00000000e+00]]


### 3.1 Trapezoidal
$\text{Area} = \frac{1}{2}(b-a)(f(a)+f(b))$ \
$\text{Let } b-a = dx$ \

$\frac{dx}{2}(f(0) + f(1) + f(1) + f(2) + f(2) + f(3)$

In [72]:
def TRAPEZOIDAL(f,a,b,n):
    dx = (b-a)/n
    sum = f(a)+f(b)

    for i in range(1, n):
        sum += 2*f(a+dx*i)
    return sum*dx*0.5

def f(x):
    return x*x
print(TRAPEZOIDAL(f, 0, 1, 4))

0.34375


### 3.2 Midpoint

In [1]:
def Midpoint_Method(f,a,b,n):
    dx = (b-a)/n
    sum = 0
    for i in range(1,n):
        m = a+dx*(0.5+i)
        sum += f(m)
    return sum*dx

### 3.3 Simpson's Rule

In [ ]:
def Simson(f,a,b,n):
    if n%2==1 : 
        return
    dx = (b-a)/n
    sum = f(a)+f(b)
    for i in range(n):
        x = a+i*dx
        if (i % 2 == 0):
            sum += 2*f(x)
        else:
            sum += 4*f(x)
    return h/3 * sum

In [ ]:
def Simson(f,a,b,n):
    if n % 2 == 1:
        return np.NaN
    dx = (b-a)/n
    sum = f(a)+f(b)
    for i in range(1,n):
        x = a+i*dx
        if i % 2 == 0:
            sum += 2*f(x)
        else
            sum += 4*f(x)
    return h/3*sum

### 4.1 Richardson's Extrapolation

In [111]:
def RICHARDSON(f,x,h,n):
    R = np.zeros((n,n))
    #fill initial with CD
    for i in range(n):
        h_i = h*(3**(-i))
        R[i,0] = (f(x+h_i)-f(x-h_i))/(2*h_i)

        for j in range(1,n):
            for i in range(j, n):
                R[i,j] = ((9**j)*R[i,j-1] - R[i-1, j-1])/(9**j-1) 
    print(R)
    return R[n-1,n-1]

print(round(RICHARDSON(lambda x : np.sin(x), 1, 0.1, 5),5))

[[0.53940225 0.         0.         0.         0.        ]
 [0.54020226 0.54030226 0.         0.         0.        ]
 [0.54029119 0.54030231 0.54030231 0.         0.        ]
 [0.54030107 0.54030231 0.54030231 0.54030231 0.        ]
 [0.54030217 0.54030231 0.54030231 0.54030231 0.54030231]]
0.5403


### 4.2 Romberg integration

In [ ]:
def Romberg_Integration(f,a,b,n):
    R = np.zeros((n,n))
    #Trap
    for i in range(n):
        interval = 2**i
        R[i,0] = Trapezoid(f,a,b,interval)
        #R[i,0] = Midpoint_Method(f,a,b,interval)
        #R[i,0] = Central Difference
    for j in range(1,n):
        for i in range(j,n):
            R[i,j] = (4**(j)*R[i,j-1]-R[i-1,j-1])/(4**j-1)
    return R[n-1,n-1]